# 01 · Unity Catalog: la jerarquía y sus objetos

⏱ **12 min** · Módulo 4 · Slides de la Intro: 17 (por qué gobernar los datos) y 18 (la jerarquía)

**Objetivo:** crear cada nivel de Unity Catalog y ubicar cualquier dato con su nombre de tres niveles.

```
Metastore ─► Catálogo ─► Esquema ─► Tablas · Vistas · Volúmenes
             workspace    lab_tvmaze_m56    suscriptores
```

| Paso | Qué haces |
|---|---|
| 1 | ¿Dónde estoy? Metastore, catálogo, esquema y usuario |
| 2 | Crear un catálogo y un esquema (el ejemplo del slide 18) |
| 3 | El esquema del laboratorio |
| 4 | Una tabla |
| 5 | Una vista y un volumen |
| 6 | Documentar: comentarios y etiquetas |
| 7 | Inventario con `information_schema` |

In [0]:
%run ./00_Setup

## Paso 1 · ¿Dónde estoy?

In [0]:
%sql
SELECT current_metastore() AS metastore, current_catalog() AS catalogo,
       current_schema() AS esquema, current_user() AS usuario

In [0]:
%sql
SHOW CATALOGS

👀 **Qué observar:** el **metastore** ya existe y está asignado a tu workspace (en Azure lo crea el administrador de la cuenta: son
los pasos 1 y 2 del slide 19). Dentro hay tres catálogos: `workspace` (el tuyo), `system` (auditoría y linaje) y `samples` (datos de ejemplo).

## Paso 2 · Crear un catálogo y un esquema

El slide 18 nombra una tabla así: `prod.retail.pedidos` → catálogo `prod`, esquema `retail`, tabla `pedidos`.
Lo construimos nivel por nivel (con el catálogo `lab_prod`, que al final borramos).

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS lab_prod COMMENT 'Catálogo de práctica: entorno de producción'

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS lab_prod.retail COMMENT 'Tablas del área de retail'

In [0]:
%sql
CREATE OR REPLACE TABLE lab_prod.retail.pedidos AS
SELECT * FROM VALUES (1, 'Lima', 120.50), (2, 'Cusco', 80.00), (3, 'Lima', 45.90) AS t(pedido_id, ciudad, monto)

In [0]:
%sql
-- catálogo.esquema.tabla
SELECT * FROM lab_prod.retail.pedidos

In [0]:
%sql
SHOW SCHEMAS IN lab_prod

👀 **Qué observar:** con el nombre de tres niveles la tabla se encuentra desde cualquier notebook, sin importar dónde estés parado.
Lo habitual es un catálogo por entorno (`dev`, `prod`) o por área. En Azure, con almacenamiento propio, el catálogo se crea con
`MANAGED LOCATION 'abfss://...'` (paso 3 del slide 19).

In [0]:
%sql
-- Era solo para practicar: CASCADE borra el catálogo con todo lo que tiene dentro
DROP CATALOG IF EXISTS lab_prod CASCADE

## Paso 3 · El esquema del laboratorio

`00_Setup` ya creó el esquema `lab_tvmaze_m56` dentro del catálogo `workspace`. Aquí vivirá todo el laboratorio.

In [0]:
%sql
DESCRIBE SCHEMA EXTENDED lab_tvmaze_m56

👀 **Qué observar:** el esquema tiene comentario y **dueño** (tú). El dueño de un objeto puede hacer todo sobre él y dar permisos a otros.

## Paso 4 · Una tabla

Para practicar el gobierno usamos una tabla pequeña con un dato sensible: los suscriptores de la plataforma, con su email.
(Los datos de TVMaze llegan en el notebook 04.)

In [0]:
%sql
-- Por si repites el notebook: la tabla se crea de cero
DROP TABLE IF EXISTS suscriptores

In [0]:
%sql
CREATE TABLE suscriptores (
  suscriptor_id INT, nombre STRING, email STRING, pais STRING, plan STRING
)

In [0]:
%sql
INSERT INTO suscriptores VALUES
  (1, 'Ana',   'ana@ejemplo.com',   'Perú',     'premium'),
  (2, 'Luis',  'luis@ejemplo.com',  'Perú',     'básico'),
  (3, 'María', 'maria@ejemplo.com', 'Colombia', 'premium'),
  (4, 'Pedro', 'pedro@ejemplo.com', 'México',   'básico'),
  (5, 'Sofía', 'sofia@ejemplo.com', 'Colombia', 'básico')

In [0]:
%sql
SELECT * FROM suscriptores

In [0]:
# La misma tabla con su nombre completo: catálogo.esquema.tabla
print(f"{CATALOG}.{SCHEMA}.suscriptores")
display(spark.table(f"{CATALOG}.{SCHEMA}.suscriptores"))

## Paso 5 · Una vista y un volumen

In [0]:
%sql
CREATE OR REPLACE VIEW suscriptores_por_pais AS
SELECT pais, count(*) AS suscriptores FROM suscriptores GROUP BY pais

In [0]:
%sql
SELECT * FROM suscriptores_por_pais

In [0]:
%sql
SHOW VOLUMES

In [0]:
# Un volumen guarda archivos (no tablas): aquí llegarán los JSON de la API
dbutils.fs.put(f"{BASE}/hola.txt", "Los volúmenes gobiernan archivos", overwrite=True)
display(dbutils.fs.ls(BASE))

👀 **Qué observar:** dentro de un esquema conviven **tablas** (datos), **vistas** (consultas guardadas) y **volúmenes** (archivos).
La ruta de un archivo repite los tres niveles: `/Volumes/catálogo/esquema/volumen/...`.

## Paso 6 · Documentar: comentarios y etiquetas

Un dato gobernado se puede **encontrar y entender**.

In [0]:
%sql
COMMENT ON TABLE suscriptores IS 'Suscriptores de la plataforma (tabla de práctica). Contiene datos personales.'

In [0]:
%sql
ALTER TABLE suscriptores ALTER COLUMN email COMMENT 'Dato personal: no compartir'

In [0]:
%sql
ALTER TABLE suscriptores SET TAGS ('dominio' = 'clientes', 'datos_personales' = 'si')

In [0]:
%sql
DESCRIBE TABLE suscriptores

👀 **Qué observar:** abre **Catalog → workspace → lab_tvmaze_m56 → suscriptores**: verás el comentario, las etiquetas y el dueño.
Las etiquetas sirven para buscar (por ejemplo, todas las tablas con `datos_personales`).

## Paso 7 · Inventario con `information_schema`

Unity Catalog expone sus metadatos como tablas que puedes consultar.

In [0]:
%sql
SELECT table_name, table_type, table_owner, comment
FROM information_schema.tables
WHERE table_schema = 'lab_tvmaze_m56' AND table_name LIKE 'suscriptores%'

In [0]:
%sql
SELECT table_name, tag_name, tag_value
FROM information_schema.table_tags
WHERE schema_name = 'lab_tvmaze_m56' AND table_name LIKE 'suscriptores%'

## ✔ Checkpoint

- [ ] Sé qué es cada nivel: metastore → catálogo → esquema → tabla, vista o volumen
- [ ] Creé un catálogo, un esquema y una tabla, y la consulté con su nombre de tres niveles
- [ ] Documenté una tabla con comentario y etiquetas
- [ ] Encontré mis objetos en `information_schema`

**Siguiente:** `02_UC_Permisos`